In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import pykitti
from pathlib import Path

# Load KITTI data
base = Path.home() / 'SensorTrust' / 'datasets' / 'kitti'
data = pykitti.raw(base_path=str(base), date='2011_09_26', drive='0009')

# Load proxies
from src.proxies.gps_proxy import extract_all_gps_proxies
from src.proxies.imu_proxy import extract_all_imu_proxies
from src.proxies.camera_proxy import extract_all_camera_proxies
from src.proxies.lidar_proxy import extract_all_lidar_proxies

dt = 0.1035

gps = extract_all_gps_proxies(data.oxts, dt=dt)
imu = extract_all_imu_proxies(data.oxts, dt=dt)

# Camera: load first 50 frames as numpy arrays
camera_frames = [np.array(data.get_cam2(i)) for i in range(min(50, len(data.cam2_files)))]
cam = extract_all_camera_proxies(camera_frames)

# LiDAR: load first 50 scans as numpy arrays
velo_scans = [data.get_velo(i) for i in range(min(50, len(data.velo_files)))]
lidar = extract_all_lidar_proxies(velo_scans, data.oxts)

print("All proxies loaded.")
print(f"gps keys: {list(gps.keys())}")
print(f"imu keys: {list(imu.keys())}")
print(f"cam keys: {list(cam.keys())}")
print(f"lidar keys: {list(lidar.keys())}")

## Normalization

In [ ]:
from src.features.normalization import MotionNormalizer

normalizer = MotionNormalizer()
normalizer.load('../src/models/normalization.pkl')
z = normalizer.transform(gps, imu, cam, lidar)

print("Normalization applied.")

## F2 + GMIS

In [ ]:
from src.features.f2_scene import compute_f2, get_f2_stats
from src.features.gmis import compute_gmis, get_gmis_stats

# Trim all proxies to the same length (LiDAR is the shortest: 49)
min_len = min(
    len(z['gps_delta_v']),
    len(z['imu_delta_v']),
    len(z['lidar_icp']),
    len(z['camera_flow'])
)

zg = z['gps_delta_v'][:min_len]
zi = z['imu_delta_v'][:min_len]
zl = z['lidar_icp'][:min_len]
zc = z['camera_flow'][:min_len]

print(f"All proxies trimmed to {min_len} frames\n")

# Compute F2
f2 = compute_f2(zg, zl)
get_f2_stats(f2)

print()

# Compute GMIS
gmis = compute_gmis(zg, zi, zl, zc)
get_gmis_stats(gmis)

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))
ax1.plot(f2)
ax1.set_ylabel('F2')
ax1.set_title('F2: GPS-LiDAR Scene Consistency')
ax1.grid(True, alpha=0.3)
ax2.plot(gmis)
ax2.set_ylabel('GMIS')
ax2.set_xlabel('Frame')
ax2.set_title('GMIS: Global Motion Inconsistency Score')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()